In [ ]:
source("helpers/init.r")
source("helpers/actionable_rules.r")
source("helpers/oncoplot_utils.r")

In [ ]:
cohorts    <- fread(paste0(READY_DATA, "cohorts.txt"))
plt_cohort <- cohorts %>% filter(!ref) %>% pull(cohort) %>% unique()
ref_cohort <- cohorts %>% filter(ref)  %>% pull(cohort) %>% unique()

# 0 - Load data

In [ ]:
protect         <- runQuery("actionability")
validate_columns(protect, c("sampleId", "top_level", "event"), "actionability")
evidence_levels <- c("FDA Approved", "Drug Repurposing",
                     "Experimental", "Other", "None")

ready <-
  cohorts %>%
  left_join(protect, by = "sampleId", relationship = "many-to-many") %>%
  mutate(top_level = case_match(top_level, "Experimental/Other Guidelines" ~ "Experimental", .default = top_level),
         top_level = ifelse(is.na(top_level), "None", top_level),
         event     = ifelse(is.na(event),     "None", event)) %>%
  select(sampleId, cohort, event, top_level) %>%
  mutate(top_level = factor(top_level, levels = evidence_levels)) %>%
  group_by(sampleId, cohort, event, top_level) %>%
  mutate(rk = row_number(top_level)) %>%
  filter(rk == 1) %>%
  ungroup()

# 1 - Rank events

In [ ]:
ref <-
  ready %>%
  group_by(cohort, event) %>%
  summarise(ct = sum(top_level != "None"), .groups = "drop") %>%
  group_by(cohort) %>%
  mutate(rk = row_number(desc(ct))) %>%
  select(-ct) %>%
  ungroup()

# 2 - Build sample × event grid

In [ ]:
ref_dt   <- as.data.table(ref)
ready_dt <- as.data.table(ready)
samps_dt <- unique(ready_dt[, .(sampleId, cohort)])

tall_ref_dt    <- ref_dt[samps_dt, on = "cohort", allow.cartesian = TRUE]
events_dt      <- unique(ready_dt[, .(sampleId, cohort, event, top_level)])
actionable_base <- merge(tall_ref_dt, events_dt,
                         by = c("sampleId", "cohort", "event"), all.x = TRUE)
actionable_base[is.na(top_level), top_level := "None"]
actionable_base <- as_tibble(actionable_base)

# 3 - Sort samples

# 4 - Prepare cohort data

In [ ]:
cohort_data    <- actionable_base %>% filter(cohort == plt_cohort, rk <= 27)
actionable_ready <- order_oncoplot(cohort_data, entity_col = "event", status_col = "top_level", none_val = "None")

# 5 - Labels

In [ ]:
prep_labels <- function() {
  base <- actionable_base %>% filter(cohort == plt_cohort, event != "None")
  ref  <- actionable_base %>% filter(cohort == ref_cohort,
                                      event %in% base$event)
  bind_rows(base, ref) %>% drop_na(event)
}

label_base     <- prep_labels()
label_pcts     <- get_label_pcts(label_base, "event", "None", "top_level")
highlights     <- get_highlights(label_base, "event", "None", "top_level")

actionable_labels <-
  bind_rows(label_pcts, highlights) %>%
  mutate(field2 = factor(
    ifelse(field != "Direction", paste0(field, " (%)"), field),
    levels = c(paste0(plt_cohort, " (%)"), paste0(ref_cohort, " (%)"), "Direction")
  ))

# 7 - Actionable oncoplot

In [ ]:
onco_actionable <-
  ggplot(actionable_ready %>% filter(top_level != "None"),
         aes(x = sampleId, y = event, fill = top_level)) +
  geom_tile() +
  scale_fill_manual(values = unlist(actionable_fill_map),
                    breaks = setdiff(names(actionable_fill_map), "None")) +
  labs(x = paste0(n_distinct(actionable_ready$sampleId[actionable_ready$top_level != "None"]),
                  " Patients")) +
  scale_x_discrete(position = "top") +
  onco_theme +
  guides(fill = guide_legend(nrow = 1))

# 6 - Labels figure

In [ ]:
labels_fig <-
  actionable_labels %>%
  filter(event %in% levels(actionable_ready$event)) %>%
  mutate(event = factor(event, levels = levels(actionable_ready$event))) %>%
  filter(!is.na(event)) %>%
  {ggplot(., aes(x = field2, y = event, label = val)) + geom_text(size = 3) + scale_x_discrete(position = "top", guide = guide_axis(n.dodge = 2)) + label_theme}

In [ ]:
# ── Unit test: Experimental recode ───────────────────────────────────────────
local({
  mock <- data.frame(
    top_level = c("FDA Approved", "Experimental/Other Guidelines", "Other", NA),
    stringsAsFactors = FALSE
  )
  result <- mock %>%
    mutate(top_level = case_match(top_level, "Experimental/Other Guidelines" ~ "Experimental", .default = top_level),
           top_level = ifelse(is.na(top_level), "None", top_level))
  stopifnot(!any(result$top_level == "Experimental/Other Guidelines"))
  stopifnot("Experimental" %in% result$top_level)
  stopifnot(!any(is.na(result$top_level)))
  message("\u2713 recode: Experimental/Other Guidelines -> Experimental; NA -> None")
})

In [ ]:
# ── Unit test: oncoplot and label events must match ───────────────────────────
events_onco  <- setdiff(levels(actionable_ready$event), "None")
events_label <- unique(labels_fig$data$event)

missing_in_label <- setdiff(events_onco,  events_label)
missing_in_onco  <- setdiff(events_label, events_onco)

if (length(missing_in_label) > 0 || length(missing_in_onco) > 0) {
  stop(sprintf(
    "Event mismatch:\n  In onco but not labels: %s\n  In labels but not onco: %s",
    paste(missing_in_label, collapse = ", "),
    paste(missing_in_onco,  collapse = ", ")
  ))
}
message(sprintf("✓ onco/labels: %d events aligned", length(events_onco)))


# 7 - Save figures

In [ ]:
figures <- list(
  onco   = onco_actionable,
  labels = labels_fig
)
saveRDS(figures, paste0(READY_FIG, "actionable.rds"))